# Sección 8. Combinación de DataFrames

En esta sección exploraremos la posibilidad de combinar información de DataFrames separados en un único DataFrame. Para ello veremos distintas formas de combinar los DataFrames de manera que la información conserve su coherencia e integración dentro de ese proceso.

## 8.1. Concatenación de DataFrames en Pandas

__Def. Concatenar__ DataFrames significa unir estructuras de datos a lo largo de un eje. 

Los DataFrames, como ya hemos visto, son estructuras bidimensionales por lo que tendremos dos opciones para concatenar:

* Unir filas -> Concatenación vertical
* Unir columnas -> Concatenación horizontal

La concatenación resulta útil cuando tenemos datos iguales separados por años o meses, queremos combinar tablas relacionadas o necesitamos añadir nuevos datos a un dataset.

Imaginemos por ejemplo que tenemo un DataFrame con los datos de ventas de la región sur y otro con los datos de ventas de la región norte:

In [ ]:
import pandas as pd

df_norte = pd.DataFrame({
    "ciudad": ["Oviedo", "Avilés", "A Coruña"],
    "ventas": [200, 150, 180]
})

df_sur = pd.DataFrame({
    "ciudad": ["Sevilla", "Cádiz", "Málaga", "Córdoba"],
    "ventas": [120, 140, 130, 90]
})

display(df_norte)
display(df_sur)

En este caso la concatenación que tiene sentido es la concatenación vertical, es decir, crear un único DataFrame con las columnas `ciudad` y `ventas` formadas por la combinación de dichas columnas en cada tabla:

In [ ]:
df_ciudad_concat = pd.concat([df_norte, df_sur])
df_ciudad_concat

__Nota.__ Es importante observar que aunque la concatenación se ha realizado con éxito hemos conservado los índices lo que ha provocado que el índice ya no sea único. 

Si deseamos evitar esto podemos utilizar el argumento `ignore_index=True`, lo que generará un nuevo índice continuo:

In [ ]:
df_ciudad_concat = pd.concat([df_norte, df_sur], ignore_index=True)
df_ciudad_concat

Si los DataFrames no comparten las mismas columnas, la concatenación se produce igualmente y se rellenan con valores nulos los espacios donde no contamos con información:

In [ ]:
df_norte["actualizado"] = True
df_sur["se_actualiza"] = False
df_ciudad_act_concat = pd.concat([df_norte, df_sur], ignore_index=True)
df_ciudad_act_concat

Análogamente podemos realizar concatenaciones horizontales con el mismo método agregando el argumento `axis=1`:

In [ ]:
df_precio = pd.DataFrame({
    "precio": [10, 12, 11]
})
df_precio

In [ ]:
df_ciudad_precio_concat = pd.concat([df_norte, df_precio], axis=1)
df_ciudad_precio_concat

El procedimiento en caso de desigualdad entre DataFrames es análogo al anterior rellenando los huecos con valores perdidos:

In [ ]:
df_ciudad_precio_sur_concat = pd.concat([df_sur, df_precio], axis=1)
df_ciudad_precio_sur_concat

Además, es interesante observar que podemos concatenar tantos DataFrames como deseemos:

In [ ]:
df_norte = pd.DataFrame({
    "ciudad": ["Oviedo", "Avilés", "A Coruña"],
    "ventas": [200, 150, 180]
})

df_sur = pd.DataFrame({
    "ciudad": ["Sevilla", "Cádiz", "Málaga", "Córdoba"],
    "ventas": [120, 140, 130, 90]
})

df_levante = pd.DataFrame({
    "ciudad": ["Barcelona", "Benidorm", "Girona"],
    "ventas": [110, 210, 160]
})

df_concat = pd.concat([df_norte, df_sur, df_levante], ignore_index=True)
df_concat

Cerraremos esta clase viendo cómo podemos gestionar los distintos tipos de unión mediante el argumento `join`.

Si indicamos `join="outer"` realizaremos una unión externa, es decir, conservaremos todas las columnas presentes en los DataFrames rellenando con nulos aquellos valores desconocidos. Esta es la opción programa por defecto

In [ ]:
df_norte["actualizado"] = True
df_sur["se_actualiza"] = False
df_ciudad_act_concat_out = pd.concat([df_norte, df_sur], ignore_index=True, join="outer")
df_ciudad_act_concat_out

Sin embargo, si indicamos `join="inner"` la unión será interna y por tanto conservaremos únicamente las columnas comunes, aquellas que se encuentran presentes en ambos DataFrames:

In [ ]:
df_ciudad_act_concat_in = pd.concat([df_norte, df_sur], ignore_index=True, join="inner")
df_ciudad_act_concat_in

### Ejercicios prácticos

* Dados los siguientes DataFrames:

In [ ]:
import pandas as pd

data_madrid = {
    "edificio": [
        "Torre Picasso",
        "Torre de Cristal",
        "Torre Cepsa"
    ],
    "ciudad": [
        "Madrid",
        "Madrid",
        "Madrid"
    ],
    "altura_metros": [
        157,
        249,
        248
    ],
    "anio_construccion": [
        1988,
        2009,
        2009
    ]
}

df_madrid = pd.DataFrame(data_madrid)
display(df_madrid)


data_barcelona = {
    "edificio": [
        "Torre Glòries",
        "Hotel Arts Barcelona",
        "Torre Mapfre"
    ],
    "ciudad": [
        "Barcelona",
        "Barcelona",
        "Barcelona"
    ],
    "altura_metros": [
        144,
        154,
        154
    ],
    "anio_construccion": [
        2005,
        1994,
        1992
    ]
}

df_barcelona = pd.DataFrame(data_barcelona)
display(df_barcelona)

data_info = {
    "plantas": [45, 50, 49, 38, 44, 40,38],
    "uso": [
        "Oficinas",
        "Oficinas",
        "Oficinas",
        "Oficinas",
        "Hotel",
        "Oficinas",
        "Otros"
    ]
}

df_info = pd.DataFrame(data_info)
display(df_info)



- Combinar `df_madrid` y `df_barcelona` para crear un nuevo DataFrame llamado `df_edificios`:

- Realiza la misma concatenación reiniciando el índice:

- Añade las columnas de `df_info` al DataFrame anterior y comprueba las dimensiones finales:

En esta clase hemos estudiado la concatenación de DataFrames tanto en horizontal como en vertical. Es interesante ver que para la concatenación horizontal estamos asumiendo que la información se encuentra ordenada. En la siguiente clase estudiaremos el método `.merge()` que nos permite realizar esta operación si los datos no están ordenados o presentan duplicados.

## 8.2. Unión de DataFrames con `.merge()`

En esta clase vamos a explorar la unión de DataFrames mediante claves utilizando el método `.merge()` para combinar las tablas. Además exploraremos los distintos tipos de unión y cómo interpretar correctamente sus resultados.

### Introducción a `.merge()`

El método `.merge()` permite combinar DataFrames utilizando una o varias columnas comunes. Esto resulta especialmente útil cuando tenemos datos repartidos en distintas tablas y necesitamos combinarlos conservando la coherencia de la información.

Veamos un ejemplo de esta idea. Supongamos que tenemos de un lado la tabla con los datos de clientes:


In [ ]:
import pandas as pd

clientes = pd.DataFrame({
    "cliente_id": [1, 2, 3, 4],
    "nombre": ["Ana", "Luis", "Carlos", "Marta"]
})

clientes

Y por otro lado tenemos las compras:

In [ ]:
compras = pd.DataFrame({
    "cliente_id": [1, 2, 2, 3, 5],
    "producto": ["Portátil", "Tablet", "Monitor", "Teclado", "Auriculares"],
    "precio": [1200, 300, 200, 50, 120]
})

compras

Observando las tablas vemos que la columna `cliente_id` es la que nos permite saber que por ejemplo, Ana se ha comprado un portátil o Marta no se ha comprado nada.

Veamos cómo podemos usar `.merge()` para poder cruzar la información y obtener una tabla que la resuma:

In [ ]:
data_cruzada = pd.merge(clientes, compras, on="cliente_id")
data_cruzada

En este caso hemos obtenido una tabla que cruza la información de todos los clientes que compran generando por ejemplo un duplicado para Luis.

__Nota.__ Es importante observar que el cliente Marta no aparece en la segunda tabla porque se han conservado únicamente los registros que se encuentran presentes en ambas tablas. Esto se debe al tipo de join que hemos realizado.

### Tipos de join

El parámetros `how` nos permite definir el tipo de merge que deseamos realizar y viene por defecto a `inner`. Las posibles opciones son:

| Tipo    | Descripción                             |
| ------- | --------------------------------------- |
| `inner` | solo coincidencias                      |
| `left`  | todas las filas del DataFrame izquierdo |
| `right` | todas las filas del DataFrame derecho   |
| `outer` | todas las filas de ambos                |

__Nota.__ Esta operación y sus tipos es análoga al join presente en SQL.

Veamos la diferencia entre los tipos de merge en la práctica. Ya hemos visto cuál sería el resultado de un inner join en la ejecución anterior. Su opuesto sería un outer que conserva todas las filas de ambos DataFrames:

In [ ]:
pd.merge(clientes, compras, how="outer")

Observamos que los campos que no tienen correspondencia en la otra tabla se rellenan con valores nulos.

Si realizamos un merge left, conservaremos todos los campos de la primera tabla y solo los que tengan correspondencia de la segunda:

In [ ]:
pd.merge(clientes, compras, how="left")

Recíprocamente si realizamos un merge right conservaremos todos los campos de la segunda tabla y solo los de la primera con los que se establezcan correspondencias:

In [ ]:
pd.merge(clientes, compras, how="right")

### Otros parámetros relevantes

En ocasiones podemos encontrarnos que aunque la información se corresponda el campo por el que se cruza no tiene el mismo nombre, en ese caso podemos usar los argumentos `left_on`y `right_on` para que el método sea capaz de establecer la conexión:

In [ ]:
antiguedad = pd.DataFrame({
    "num_cliente": [1, 2, 3, 4, 5],
    "meses": [36, 14, 100, 2, 60]
})

antiguedad

In [ ]:
pd.merge(clientes, antiguedad, how="inner", left_on="cliente_id", right_on="num_cliente")

Por último en ocasiones querremos realizar cruces por más de una columna. Esto es posible mediante el argumento `on` que puede tomar como valores un único elemento (una columna) o una lista de elementos (varias columnas):

In [ ]:
df1 = pd.DataFrame({
    "ciudad": ["Madrid", "Madrid", "Barcelona", "Valencia"],
    "producto": ["Manzana", "Naranja", "Manzana", "Pera"],
    "ventas": [100, 150, 200, 120]
})

df1

In [ ]:
df2 = pd.DataFrame({
    "ciudad": ["Madrid", "Barcelona", "Barcelona", "Valencia"],
    "producto": ["Manzana", "Manzana", "Pera", "Pera"],
    "precio": [1.2, 1.1, 1.5, 1.3]
})
df2

In [ ]:
pd.merge(df1, df2, on=["ciudad", "producto"], how="inner")

### Ejercicio práctico

* Dados los siguientes DataFrames:

In [ ]:
df_temp = pd.DataFrame({
    "ciudad": ["Madrid", "Barcelona", "Sevilla", "Bilbao", "Valencia", "Málaga"],
    "pais": ["España", "España", "España", "España", "España", "España"],
    "fecha": ["2025-01-10", "2025-01-10", "2025-01-10", "2025-01-10", "2025-01-10", "2025-01-10"],
    "temperatura_media": [8.5, 10.2, 12.8, 7.1, 11.0, 13.4]
})

df_temp

In [ ]:
df_lluvia = pd.DataFrame({
    "ciudad": ["Madrid", "Barcelona", "Sevilla", "Zaragoza", "Valencia", "Málaga"],
    "pais": ["España", "España", "España", "España", "España", "España"],
    "fecha": ["2025-01-10", "2025-01-10", "2025-01-10", "2025-01-10", "2025-01-10", "2025-01-10"],
    "precipitacion_mm": [2.1, 0.0, 1.5, 0.3, 0.8, 0.0]
})

df_lluvia

In [ ]:
df_humedad = pd.DataFrame({
    "localidad": ["Madrid", "Barcelona", "Bilbao", "Valencia", "Vigo", "Madrid", "Barcelona",],
    "fecha_medicion": ["2025-01-10", "2025-01-10", "2025-01-10", "2025-01-10", "2025-01-10", "2025-01-12", "2025-01-12"],
    "humedad_relativa": [71, 68, 82, 65, 79, 70, 67]
})

df_humedad

* Construye un DataFrame que contenga solo las ciudades para las que existen datos tanto de temperatura como de precipitación.

* Crea un DataFrame que mantenga todas las ciudades del dataset de temperaturas, aunque no tengan datos de precipitación:

* Construye un DataFrame que contenga todas las ciudades presentes en cualquiera de los dos datasets, aunque no haya datos en el otro.
Identifica qué ciudades tienen valores faltantes.

* Combina los datos de temperatura con los datos de humedad:

En esta clase hemos explorado como `.merge()` nos permite unir DataFrames preservando la integridad de la información incluso si las columnas tienen duplicados o nombres distintos.

En la siguiente clase estudiaremos el método `.join()` para unir DataFrames a partir de la información almacenada en los índices.

## 8.3. Unión de dataframes con `.join()`

El método `.join()` permite combinar dos DataFrames utilizando los índices como clave de unión en lugar de una columna del DataFrame como en el caso del `.merge()`.

Este método es especialmente útil cuando los DataFrames ya están indexados y queremos añadir nuevas columnas.

Veamos a continuación un ejemplo de cómo podemos hacer esto en Pandas:


In [ ]:
import pandas as pd

clientes = pd.DataFrame({
    "nombre": ["Ana", "Luis", "Carlos", "Marta"],
    "ciudad": ["Madrid", "Sevilla", "Valencia", "Madrid"]
}, index=["A189", "B145", "C643", "D759"])

clientes

In [ ]:
compras = pd.DataFrame({
    "producto": ["Laptop", "Tablet", "Monitor"],
    "precio": [1200, 300, 200]
}, index=["A189", "B145", "C643"])

compras

Los índices en este caso representan códigos identificadores de clientes. Procedemos a unir los DataFrames con el índice como clave:

In [ ]:
clientes.join(compras)

__Nota.__ Observamos que Marta no tiene ninguna compra registrada y aparece en la tabla resultado.

Esto se debe a que en el caso del método `.join()` disponemos de los mismos tipos de cruces que en `.merge()` pero en el caso de `.join()` el tipo elegido por defecto es `how="left"`.

Podemos explorar los distintos tipos de cruces igual que hicimos con `.merge()`:

In [ ]:
clientes.join(compras, how="inner")

In [ ]:
clientes.join(compras, how="right")

In [ ]:
clientes.join(compras, how="outer")

La clave a la hora de elegir si debemos usar `.merge()` o `join()` consiste en la estructura de nuestros datos. `join()` requiere una determinada estructura y es menos flexible pero a cambio permite una sintaxis más limpia.

Además, resulta bastante habitual que el índice de un DataFrame sea una clave identificativa que permite cruzarlo con otros.

Es interesante ver que además podemos unir varios DataFrames en una misma sentencia:

In [ ]:
antiguedad = pd.DataFrame({
    "meses": [60, 14, 26]
}, index=["A189", "B145", "C643"])

antiguedad

In [ ]:
df_join_completo = clientes.join([compras, antiguedad])
df_join_completo

Por último, con `join()` también podemos cruzar utilizando una columna:

In [ ]:
df1 = pd.DataFrame({
    "pedido_id": [101, 102, 103, 104],
    "cliente_id": [1, 2, 3, 2],
    "producto": ["Laptop", "Ratón", "Teclado", "Monitor"]
})
df1

In [ ]:
df2 = pd.DataFrame({
    "cliente_id": [1, 2, 3],
    "nombre": ["Ana", "Luis", "Marta"],
    "ciudad": ["Madrid", "Barcelona", "Valencia"]
})
df2

In [ ]:
df1.join(df2.set_index("cliente_id"), on="cliente_id")

`df2.set_index("cliente_id")` convierte `cliente_id` en índice.

`join(..., on="cliente_id")` usa la columna `cliente_id` de df1 para buscar coincidencias en el índice de df2.

### Ejercicio práctico

* Dados los siguientes DataFrames:

In [ ]:
df_peliculas = pd.DataFrame({
    "id_pelicula": [1, 2, 3, 4, 5],
    "titulo": [
        "Inception",
        "The Godfather",
        "Parasite",
        "The Dark Knight",
        "Pulp Fiction"
    ],
    "anio": [2010, 1972, 2019, 2008, 1994]
}).set_index("id_pelicula")

df_peliculas

In [ ]:
df_directores = pd.DataFrame({
    "id_pelicula": [1, 2, 3, 4],
    "director": [
        "Christopher Nolan",
        "Francis Ford Coppola",
        "Bong Joon-ho",
        "Christopher Nolan"
    ]
}).set_index("id_pelicula")
df_directores

In [ ]:
df_puntuacion = pd.DataFrame({
    "id_pelicula": [1, 2, 3, 4, 6],
    "puntuacion_imdb": [8.8, 9.2, 8.5, 9.0, 7.5]
}).set_index("id_pelicula")

df_puntuacion

In [ ]:
df_presupuesto = pd.DataFrame({
    "titulo": [
        "Inception",
        "The Godfather",
        "Parasite",
        "The Dark Knight",
        "Pulp Fiction"
    ],
    "presupuesto_millones": [160, 6, 11, 185, 8]
})
df_presupuesto

* Añade al DataFrame de películas la columna director:

* Añade ahora la columna puntuacion_imdb al DataFrame de películas:

* Crea un DataFrame que contenga películas, directores y puntuaciones combinando los tres datasets:

* Crea un nuevo DataFrame que contenga solo las películas que tengan puntuación registrada:

En esta clase hemos visto como `.join()` es el equivalente a `.merge()` cuando deseamos utilizar el índice como clave de unión. 

En la última clase de esta sección exploraremos distintos problemas que suelen surgir cuando se trabaja con estas uniones y posibles soluciones para ellos.

## 8.4. Resolución de conflictos en uniones 

Cuando trabajamos con métodos para la unión de distintos DataFrames afrontamos distintos problemas que pueden dificultar la interpretación del resultado final o incluso afectar a la coherencia de los datos resultantes. A continuación, recorreremos los principales problemas y algunas posibles soluciones para estas situaciones.

Partiremos de los siguientes DataFrames de ejemplo:

In [ ]:
import pandas as pd

empleados = pd.DataFrame({
    "empleado_id": [1, 2, 3],
    "nombre": ["Ana", "Luis", "Carlos"],
    "ciudad": ["Madrid", "Sevilla", "Valencia"]
})

empleados

In [ ]:
salarios = pd.DataFrame({
    "empleado_id": [1, 2, 4],
    "ciudad": ["Madrid", "Sevilla", "Barcelona"],
    "salario": [2500, 3000, 2800]
})

salarios

### Columnas duplicadas

Si realizamos un `.merge()` observamos que Pandas añade automáticamente el sufijo `_x` y `_y` a las columnas duplicadas para permitirnos distinguir el origen de cada una:

In [ ]:
pd.merge(empleados, salarios, on="empleado_id")

La solución más evidente a este problema sería renombrar la columna en uno de los dos DataFrames:

In [ ]:
salarios_renom = salarios.copy()
salarios_renom = salarios_renom.rename(columns={
    "ciudad" : "city"
})


In [ ]:
salarios_renom

In [ ]:
pd.merge(empleados, salarios_renom, on="empleado_id")

Otra posibilidad es personalizar los sufijos para conservar los DataFrames iniciales pero tener trazabilidad en el DataFrame de unión:

In [ ]:
pd.merge(
    empleados,
    salarios,
    on="empleado_id",
    suffixes=("_empleado", "_salario")
)

### Detección del origen de los valores

En ocasiones y sobre todo cuando no estamos tan acostumbrados a trabajar con la unión de DataFrames o no tenemos claro el tipo de unión que necesitaremos puede ser útil identificar el origen de cada registro del DataFrame resultante.

Para ello podemos usar el argumento `indicator=True` que crea una columna extra llamada `_merge` que nos indica si la fila resultante proviene de un DataFrame, del otro o de ambos:

In [ ]:
pd.merge(
    empleados,
    salarios,
    on="empleado_id",
    how="outer",
    indicator=True
)

Esta función es además muy útil para llevar a cabo auditorías de datos, localizando por ejemplo, para qué registros no tenemos información en todas las tablas cuando sí deberíamos tenerlos.

### Claves duplicadas

La columna que utilicemos como clave de unión no tiene porque ser una columna de valores únicos. Es decir, podemos tener valores duplicados que conducirían a la generación de múltiples filas. Esto no es un problema per sé pero en ocasiones cuando trabajamos con datos de gran tamaño es importante tenerlo en cuenta para evitar que las tablas resultantes sean de un tamaño innecesariamente grande que puedan llegar a generar conflictos de recursos en su procesado.

Es una una buena práctica proceder al filtrado antes que al cruce para cruzar tablas lo más pequeñas posibles:

In [ ]:
salarios = pd.DataFrame({
    "empleado_id": [1, 2, 2, 3, 3],
    "ciudad": ["Madrid", "Sevilla", "Sevilla", "Barcelona", "Barcelona"],
    "salario": [2500, 3000, 2800, 850, 900]
})

salarios

In [ ]:
cruce_empleado_salario = pd.merge(
    empleados,
    salarios,
    on="empleado_id",
    how="left"
)
cruce_empleado_salario

In [ ]:
print(empleados.shape, cruce_empleado_salario.shape)

Vemos que aún tratándose de un left join hemos obtenido un DataFrame con más registros que el DataFrame "izquierdo" original.

Para gestionar este tipo de situaciones contamos con el argumento `validate` que antes de realizar el merge comprueba si las claves son únicas o no.

Este argumento puede tomar los siguientes valores:

| opción         | significado                       |
| -------------- | --------------------------------- |
| `one_to_one`   | claves únicas en ambos DataFrames |
| `one_to_many`  | izquierda única                   |
| `many_to_one`  | derecha única                     |
| `many_to_many` | múltiples en ambos                |


In [ ]:
pd.merge(
    empleados,
    salarios,
    on="empleado_id",
    how="left", 
    validate="one_to_one"
)


In [ ]:
pd.merge(
    empleados,
    salarios,
    on="empleado_id",
    how="left", 
    validate="one_to_many"
)


Cuando vayamos a realizar cruces de DataFrames es recomendable:

* Revisar las claves antes de la unión para evitar duplicados innecesarios.
* Verificar claramente las relaciones entre tablas y añadir validadores si lo consideramos necesarios
* Usar `indicator=True` al principio cuando tengamos dudas con un merge determinado.
* Renombrar columnas conflictivas de forma directa o mediante el uso de sufijos.

### Ejercicio práctico

* Dados los siguientes DataFrames:

In [ ]:
df_productos = pd.DataFrame({
    "id_producto": [101, 102, 103, 104, 105, 106],
    "producto": ["Manzana", "Leche", "Pan", "Arroz", "Yogur", "Tomate"],
    "categoria": ["Fruta", "Lácteo", "Panadería", "Cereal", "Lácteo", "Verdura"],
    "precio": [1.20, 0.95, 1.10, 2.30, 0.80, 1.50]
})
df_productos

In [ ]:
df_ventas = pd.DataFrame({
    "id_producto": [101, 101, 102, 103, 103, 103, 107],
    "fecha": [
        "2025-01-10", "2025-01-11", "2025-01-10",
        "2025-01-10", "2025-01-11", "2025-01-12", "2025-01-10"
    ],
    "unidades_vendidas": [10, 8, 15, 20, 18, 25, 12],
    "precio": [1.20, 1.25, 0.95, 1.10, 1.10, 1.15, 2.00]
})
df_ventas

In [ ]:
df_proveedores = pd.DataFrame({
    "id_producto": [101, 101, 102, 104, 105, 108],
    "proveedor": ["Proveedor A", "Proveedor B", "Proveedor C", "Proveedor D", "Proveedor E", "Proveedor F"],
    "pais_origen": ["España", "Francia", "España", "Italia", "España", "Portugal"]
})

df_proveedores

In [ ]:
df_stock = pd.DataFrame({
    "id_producto": [101, 102, 103, 104, 105],
    "stock": [50, 80, 40, 100, 60]
}).set_index("id_producto")
df_stock

* Combinar la información de ventas con el de productos y presta atención al tamaño del DataFrame resultante. Haz que el resultado permita distinguir el precio de cada tabla:

* Combinar el dato de ventas con el de proveedores. Observa de nuevo el tamaño resultante:

* Une los datos de producto y stock usando `.join()`:

La unión de DataFrames mediante distintas técnicas es un problemas bastante habitual al trabajar con datos y es importante comprender las implicaciones que puede tener elegir un método u otro así como las posibles alteraciones en los datos que pueden generar estos cruces.

En las última sección haremos un repaso general de las principales diferencias entre NumPy y Pandas para cerrar este curso.